In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
from analysis_framework import Dataset
from ReweightingHelper import ReweightingHelper
from AltSetupHandler import AltSetupHandler
from math import sin

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x80b2660
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x814b340


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 4
# prod = False
prod = True
no_rvec = True
# write_outputs = False
write_outputs = True
# dataset_path = "data/datasets/selected-objects/test.json"
# output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/oo-sqme/test"
# output_meta_path = "data/datasets/oo-sqme"
# output_meta = f"{output_meta_path}/test.json"
# checked_output_meta = f"{output_meta_path}/checked-test.json"
output_collections = r"(\w*sqme\w*)"
if prod:
    dataset_path = "data/datasets/selected-objects-new2/signal-only.json"
    output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/oo-sqme-new2/signal-only-sqrts"
    output_meta_path = "data/datasets/oo-sqme-new2"
    output_meta = f"{output_meta_path}/signal-only-sqrts.json"

In [4]:
# ROOT.EnableImplicitMT(n_threads)
environ["OMP_NUM_THREADS"] = str(n_threads)

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ReweightingHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xb845a40


In [7]:
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "mW": 80.419,
    "g1z": 1.0,
    "ka": 1.0,
    "la": 0.0
  },
"variations": [
    1e-08
  ]
}
""", mirror=False, combinations=False)
alt_configs = alt_setup_handler.get_alt_setup()
print(alt_configs)
analysis.initialise_omega_wrappers(alt_configs)

{'mW_pos_1em08': {'mW': 80.41900000999999, 'g1z': 1.0, 'ka': 1.0, 'la': 0.0}, 'g1z_pos_1em08': {'mW': 80.419, 'g1z': 1.00000001, 'ka': 1.0, 'la': 0.0}, 'ka_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.00000001, 'la': 0.0}, 'la_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.0, 'la': 1e-08}}


In [8]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [9]:
min_setups = ["nominal", "g1z_pos_1em08", "ka_pos_1em08", "la_pos_1em08"] + ["mW_pos_1em08"]

In [10]:
x_points = [-2e-3, -1e-3, -5e-4, -2.5e-4, -1e-4, 1e-4, 2.5e-4, 5e-4, 1e-3, 2e-3]

In [11]:
# define nominal beam lvecs
analysis.Define("nominal_beam_e_lvec", "ROOT::Math::PxPyPzMVector(+8.750143e-01, 0., +1.250000e+02, +5.109968e-04)")
analysis.Define("nominal_beam_p_lvec", "ROOT::Math::PxPyPzMVector(+8.750143e-01, 0., -1.250000e+02, +5.109968e-04)")

# define varied beam lvecs
for var in x_points:
    sqrts = 250. + var
    E = sqrts/2
    var_name = AltSetupHandler.make_name("", var)
    analysis.define_only_on(signal_category, f"var{var_name}_beam_e_lvec", f"ROOT::Math::PxPyPzMVector({sin(x_angle/2) * E}, 0., {E}, +5.109968e-04)")
    analysis.define_only_on(signal_category, f"var{var_name}_beam_p_lvec", f"ROOT::Math::PxPyPzMVector({sin(x_angle/2) * E}, 0., -{E}, +5.109968e-04)")
    analysis.define_only_on(signal_category, f"var{var_name}_nu_lvec", f"auto proto_lvec = var{var_name}_beam_e_lvec + var{var_name}_beam_p_lvec - true_lep_lvec - true_quark1_lvec - true_quark2_lvec; return ROOT::Math::PxPyPzEVector(proto_lvec.Px(), proto_lvec.Py(), proto_lvec.Pz(), proto_lvec.P())")

In [12]:
for var in x_points:
    var_name = AltSetupHandler.make_name("", var)
    analysis.book_sqme(
                        [
                            f"var{var_name}_beam_e_lvec",
                            f"var{var_name}_beam_p_lvec",
                            "true_lep_lvec",
                            f"var{var_name}_nu_lvec",
                            "true_quark1_lvec",
                            "true_quark2_lvec",
                        ],
                        "iso_lep_charge",
                        f"mc_E_var{var_name}",
                        alt_setups=min_setups,
                        categories=signal_category,
                        hels=True
                        )
    analysis.book_sqme(
                        [
                            f"var{var_name}_beam_e_lvec",
                            f"var{var_name}_beam_p_lvec",
                            "true_lep_lvec",
                            f"var{var_name}_nu_lvec",
                            "true_quark2_lvec",
                            "true_quark1_lvec",
                        ],
                        "iso_lep_charge",
                        f"wj_mc_E_var{var_name}",
                        alt_setups=min_setups,
                        categories=signal_category,
                        hels=True
                        )

In [13]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec, write_categories=signal_category)

Info in <[ROOT.RDF] Info /tmp/root/spack-stage/spack-stage-root-6.38.00-2jf5cmbvudyjye7uzxzvsgmmlw2msfso/spack-build-2jf5cmb/include/ROOT/RDF/RInterface.hxx:1363 in auto ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager, void>::Snapshot(std::string_view, std::string_view, const ColumnNames_t &, const RSnapshotOptions &)::(anonymous class)::operator()() const [Proxied = ROOT::Detail::RDF::RLoopManager, DataSource = void]>: 
	In ROOT 6.38, the default compression settings of Snapshot have been changed from 101 (ZLIB with compression level 1, the TTree default) to 505 (ZSTD with compression level 5). This change may result in smaller Snapshot output dataset size by default. In order to suppress this message, set 'ROOT_RDF_SNAPSHOT_INFO=0' in your environment or set 'ROOT.RDF.Snapshot.Info: 0' in your .rootrc file.


In [14]:
analysis.book_reports()

In [15]:
%%time
analysis.run()

CPU times: user 10h 27min 32s, sys: 15min 57s, total: 10h 43min 30s
Wall time: 2h 45min 38s


In [16]:
# if write_outputs:
    # analysis.check_snapshots("events", output_path, checked_output_meta)

In [17]:
analysis.print_reports()

         4f_sw_sl_signal               4f_sl_bkg
               0 (0e+00)               0 (0e+00) All
                   0e+00                   0e+00 efficiency

